# 04 SVM Aspect Classification

Executable notebook for Phase 9A and Phase 9B SVM aspect-classification experiments. SVM is used for aspect classification only; IndoBERT remains the selected sentiment model.


## CRISP-DM Stage

Modeling and evaluation. This notebook prepares the SVM candidate dataset, trains controlled SVM aspect-classifier scenarios, and compares the scenarios using weak-label evaluation metrics.


## SVM Role in SentiRank

The SVM classifier predicts actionable review aspects from `text_svm`. The predicted aspect output will later support criteria alignment for AHP/Fuzzy AHP, after expert judgement validates which criteria are appropriate.

SVM is not used for sentiment classification. Sentiment classification is handled by the final IndoBERT candidate, `run_3_weighted_loss_lr_1e-5`.


## Weak Label and Class Imbalance Limitations

The SVM labels are weak labels generated from keyword-based aspect rules, not expert-validated ground truth. Phase 9A removed `General` and low-confidence labels, but class imbalance remains severe. In the 7-class candidate dataset, `UI/UX` and `Audio Quality` are small minority classes.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "CLAUDE.md").exists() and (candidate / "ml-service").exists():
            return candidate
    raise RuntimeError("Could not find SentiRank project root from current working directory.")


PROJECT_ROOT = find_project_root()
ML_SERVICE_DIR = PROJECT_ROOT / "ml-service"
DATASETS_DIR = PROJECT_ROOT / "datasets"
EDA04_DIR = DATASETS_DIR / "outputs" / "eda" / "04_svm"
FIG04_DIR = PROJECT_ROOT / "docs" / "figures" / "04_svm"
MODEL_DIR = ML_SERVICE_DIR / "saved_models" / "svm"
INPUT_DATASET = DATASETS_DIR / "processed" / "svm" / "svm_aspect_dataset.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Input dataset: {INPUT_DATASET}")
print(f"Metrics directory: {EDA04_DIR}")
print(f"Figure directory: {FIG04_DIR}")
print(f"Model artifact directory: {MODEL_DIR}")


## Scenario Definitions

Scenario A uses the original 7 actionable aspect labels: `Features & Content`, `Ads Experience`, `Subscription & Pricing`, `Performance & Stability`, `Account/Login`, `Audio Quality`, and `UI/UX`.

Scenario B uses a merged 5-class stable taxonomy:

- `Features & Content` + `Audio Quality` -> `Features, Content & Audio Experience`
- `Performance & Stability` + `UI/UX` -> `App Reliability & Usability`
- `Ads Experience`, `Subscription & Pricing`, and `Account/Login` stay unchanged

The merge is intended to reduce extreme minority-class instability while keeping the taxonomy interpretable for later AHP/Fuzzy AHP criteria alignment.


## TF-IDF and SVM Training Strategy

The training script uses a scikit-learn pipeline with combined word and character features:

- Word TF-IDF: unigram and bigram, `min_df=2`, `max_df=0.95`, `sublinear_tf=True`
- Character TF-IDF: `char_wb`, n-gram range 3 to 5, `min_df=2`, `sublinear_tf=True`
- Classifier: `LinearSVC(class_weight="balanced", random_state=42)`

Exact duplicate `text_svm` values are deduplicated before stratified train/validation/test splitting to reduce leakage risk.


## Run SVM Training Script

This cell trains both controlled SVM scenarios. It does not train IndoBERT, modify raw datasets, fit unrelated models, or touch application/database layers.


In [ ]:
RUN_SVM_TRAINING = True

if RUN_SVM_TRAINING:
    command = [sys.executable, "scripts/train_svm_aspect_classifier.py", "--scenario", "both"]
    result = subprocess.run(command, cwd=ML_SERVICE_DIR, text=True, capture_output=True, check=False)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"SVM training failed with return code {result.returncode}")
else:
    display(Markdown("SVM training skipped. Set `RUN_SVM_TRAINING = True` to regenerate Phase 9B results."))


## Load Scenario Comparison

Scenario selection is based on higher macro F1, better minority-class stability, interpretable taxonomy, AHP/Fuzzy AHP alignment, and lower risk from extreme class imbalance.


In [ ]:
def load_json(path: Path):
    if not path.exists():
        display(Markdown(f"Missing JSON: `{path.relative_to(PROJECT_ROOT)}`"))
        return None
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def load_csv(path: Path):
    if not path.exists():
        display(Markdown(f"Missing CSV: `{path.relative_to(PROJECT_ROOT)}`"))
        return None
    return pd.read_csv(path)


comparison = load_csv(EDA04_DIR / "svm_scenario_comparison.csv")
training_summary = load_json(EDA04_DIR / "svm_training_summary.json")

if comparison is not None:
    display(comparison)
if training_summary:
    display(Markdown(f"Selected candidate scenario: `{training_summary['candidate_selection']['selected_scenario']}`"))
    display(pd.json_normalize(training_summary["candidate_selection"]["selected_metrics"]).T.reset_index().rename(columns={"index": "metric", 0: "value"}))


## Classification Reports

Reports are saved for the original 7-class baseline and the merged 5-class taxonomy.


In [ ]:
for scenario in ["original_7class", "merged_5class"]:
    report = load_json(EDA04_DIR / f"svm_{scenario}_classification_report.json")
    if report:
        display(Markdown(f"### {scenario} classification report"))
        rows = []
        for label, values in report.items():
            if isinstance(values, dict):
                row = {"label": label, **values}
                rows.append(row)
        display(pd.DataFrame(rows))


## Split and Confusion Matrix Tables

Split distributions and confusion matrices are saved as CSV metrics for auditability.


In [ ]:
for scenario in ["original_7class", "merged_5class"]:
    for filename in [
        f"svm_{scenario}_split_distribution.csv",
        f"svm_{scenario}_confusion_matrix.csv",
    ]:
        table = load_csv(EDA04_DIR / filename)
        if table is not None:
            display(Markdown(f"### {filename}"))
            display(table)


## Figures

Figures are generated under `docs/figures/04_svm/` using matplotlib only.


In [ ]:
def show_figures(paths: list[Path]) -> None:
    for path in paths:
        if path.exists():
            display(Markdown(f"**{path.relative_to(PROJECT_ROOT)}**"))
            display(Image(filename=str(path)))
        else:
            display(Markdown(f"Missing figure: `{path.relative_to(PROJECT_ROOT)}`"))


show_figures([
    FIG04_DIR / "svm_original_7class_confusion_matrix.png",
    FIG04_DIR / "svm_original_7class_class_f1.png",
    FIG04_DIR / "svm_merged_5class_confusion_matrix.png",
    FIG04_DIR / "svm_merged_5class_class_f1.png",
    FIG04_DIR / "svm_scenario_comparison.png",
])


## Model Artifacts

Local artifacts are written under `ml-service/saved_models/svm/` and must not be committed to Git:

- `svm_original_7class_pipeline.joblib`
- `svm_original_7class_label_mapping.json`
- `svm_merged_5class_pipeline.joblib`
- `svm_merged_5class_label_mapping.json`


In [ ]:
artifact_paths = [
    MODEL_DIR / "svm_original_7class_pipeline.joblib",
    MODEL_DIR / "svm_original_7class_label_mapping.json",
    MODEL_DIR / "svm_merged_5class_pipeline.joblib",
    MODEL_DIR / "svm_merged_5class_label_mapping.json",
]
artifact_status = [
    {"artifact": str(path.relative_to(PROJECT_ROOT)), "exists": path.exists(), "size_bytes": path.stat().st_size if path.exists() else None}
    for path in artifact_paths
]
display(pd.DataFrame(artifact_status))


## Interpretation and Next Step

The selected SVM scenario should be interpreted as performance against weak labels, not as expert-validated aspect ground truth. The merged 5-class scenario is preferred when it improves macro F1 and minority-class stability while keeping the taxonomy interpretable.

Next step: align the selected aspect taxonomy with AHP/Fuzzy AHP criteria and expert judgement before final decision-support weighting.
